# AoU Proposal 1 V1 — Wearable × Proteomics\n\n**Primary analysis:** longitudinal Fitbit sleep/circadian phenotypes → Olink NPX proteome-wide association → CDRv9 cis-pQTL prioritization.\n\nRun only in the All of Us Researcher Workbench 2.0 **Controlled Tier**. The notebook exports aggregate protein-level statistics only.\n\nPrimary prespecified digital phenotypes are sleep duration, duration variability, sleep midpoint variability, sleep-onset variability, social jetlag, and sleep efficiency. A true minute-level Sleep Regularity Index can be added later from `sleep_level`; V1 deliberately avoids labeling a nightly-timing proxy as SRI.

In [ ]:
# All of Us Proposal 1 V1 — ONE RUN CELL
# Fitbit sleep/circadian phenotypes × Olink proteome-wide association × cis-pQTL prioritization
# Run only inside an All of Us Researcher Workbench 2.0 Controlled Tier workspace.

import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------------
WORKSPACE_CDR_OVERRIDE = None

# Prefer the CDRv9 normalized, replicate-removed proteomics TSV/Parquet.
# If left None, the notebook tries obvious environment-variable paths.
PROTEOMICS_NPX_RESOURCE = None

# Optional but strongly recommended for the causal-prioritization layer.
# Point to the CDRv9 cis-pQTL summary resource described in the genomics/multi-omics documentation.
CIS_PQTL_RESOURCE = None

MIN_SLEEP_DAYS = 21
MIN_ASSOCIATION_N = 100

DIGITAL_FEATURES = [
    "mean_sleep_hours",
    "sd_sleep_hours",
    "sleep_midpoint_variability_min",
    "sleep_onset_variability_min",
    "social_jetlag_min",
    "mean_sleep_efficiency",
]

# Primary model covariates available directly from the OMOP person table.
# Extend later with BMI/smoking/genetic ancestry when the thesis model is frozen.
COVARIATES = ["age_at_cdr_cutoff", "gender_concept_id"]

FDR_THRESHOLD = 0.05
PQTL_Q_THRESHOLD = 0.05
# -------------------------------------------------------------------------

helper_dirs = [
    Path("../research/digital_health_omics"),
    Path("research/digital_health_omics"),
]
helper_dir = next((p.resolve() for p in helper_dirs if (p/"allofus_proposal1.py").exists()), None)
if helper_dir is None:
    raise FileNotFoundError(
        "allofus_proposal1.py not found. Clone/open this notebook from kimtk94/Codex."
    )

sys.path.insert(0, str(helper_dir))

from allofus_stage0 import (
    get_cdr_ref,
    get_bq_client,
    infer_proteomics_path,
    environment_resource_hints,
)
from allofus_proposal1 import (
    load_proteomics_long,
    query_sleep_nightly,
    aggregate_sleep_phenotypes,
    query_person_covariates,
    proteome_wide_scan,
    candidate_table,
    load_cis_pqtl,
    prioritize_with_pqtl,
)

cdr = get_cdr_ref(WORKSPACE_CDR_OVERRIDE)
client = get_bq_client()

print("="*88)
print("All of Us CDRv9 Proposal 1 V1")
print("Fitbit sleep/circadian × Olink proteomics × cis-pQTL")
print("="*88)
print("WORKSPACE_CDR:", cdr)

# -------------------------------------------------------------------------
# 1. Resolve and load proteomics
# -------------------------------------------------------------------------
prot_path = PROTEOMICS_NPX_RESOURCE or infer_proteomics_path()

if not prot_path:
    print("\nCould not auto-detect a proteomics resource. Relevant workspace variables:")
    display(environment_resource_hints())
    raise RuntimeError(
        "Set PROTEOMICS_NPX_RESOURCE to the CDRv9 normalized replicate-removed "
        "proteomics TSV/Parquet resource. Prefer the file exposing ResearchID."
    )

proteomics, prot_meta = load_proteomics_long(prot_path)
print("\nProteomics metadata:")
print(json.dumps(prot_meta, indent=2))
print(f"Proteomics participants: {proteomics['person_id'].nunique():,}")
print(f"Protein assays:          {proteomics['protein'].nunique():,}")

# Restrict expensive Fitbit query to participants who actually have proteomics.
protein_ids = sorted(proteomics["person_id"].astype(str).unique())

# -------------------------------------------------------------------------
# 2. Extract Fitbit nightly data only for the proteomics cohort
# -------------------------------------------------------------------------
print("\nQuerying nightly Fitbit sleep data for proteomics participants...")
nightly, sleep_source = query_sleep_nightly(client, cdr, protein_ids)
print("Sleep source mapping:")
print(json.dumps(sleep_source, indent=2))
print(f"Raw nightly rows: {len(nightly):,}")

sleep = aggregate_sleep_phenotypes(nightly, min_days=MIN_SLEEP_DAYS)
print(f"Participants with >= {MIN_SLEEP_DAYS} valid sleep days: {len(sleep):,}")

if sleep.empty:
    raise RuntimeError("No participants pass sleep QC; inspect source mapping / Fitbit overlap.")

# -------------------------------------------------------------------------
# 3. OMOP demographic covariates
# -------------------------------------------------------------------------
analysis_ids = sorted(set(sleep["person_id"]) & set(protein_ids))
covars = query_person_covariates(client, cdr, analysis_ids)

print("\nPrimary analysis cohort:")
print(f"Fitbit-QC ∩ Proteomics: {len(analysis_ids):,}")
print(f"Covariate rows:         {len(covars):,}")

# Descriptive summaries only; do not export participant identifiers.
print("\nDigital phenotype summary:")
desc_cols = [c for c in DIGITAL_FEATURES if c in sleep.columns]
display(sleep[desc_cols].describe().T)

# -------------------------------------------------------------------------
# 4. Proteome-wide association scan
#    NPX ~ standardized digital phenotype + age + sex
# -------------------------------------------------------------------------
print("\nRunning proteome-wide association scans...")
scan = proteome_wide_scan(
    proteomics=proteomics,
    phenotypes=sleep,
    covariates=covars,
    digital_features=DIGITAL_FEATURES,
    covariate_columns=COVARIATES,
    min_n=MIN_ASSOCIATION_N,
)

if scan.empty:
    raise RuntimeError("Association scan returned no results.")

print(f"Association tests: {len(scan):,}")
print(f"Proteins with FDR <= {FDR_THRESHOLD}: {(scan['q'] <= FDR_THRESHOLD).sum():,}")

hits = candidate_table(scan, q_threshold=FDR_THRESHOLD, top_n_per_feature=30)

print("\nTop protein associations by digital phenotype:")
display(
    hits[
        ["digital_feature","protein","olink_id","uniprot","n","beta","se","p","q","significant"]
    ].head(100)
)

# -------------------------------------------------------------------------
# 5. cis-pQTL prioritization
# -------------------------------------------------------------------------
pqtl_prioritized = pd.DataFrame()

if CIS_PQTL_RESOURCE:
    print("\nLoading CDRv9 cis-pQTL summary...")
    cis = load_cis_pqtl(CIS_PQTL_RESOURCE)
    pqtl_prioritized = prioritize_with_pqtl(
        scan,
        cis,
        scan_q=FDR_THRESHOLD,
        pqtl_q=PQTL_Q_THRESHOLD,
    )

    print(f"PWAS hits with cis-pQTL annotation rows: {len(pqtl_prioritized):,}")
    if not pqtl_prioritized.empty:
        show = [
            c for c in [
                "digital_feature","protein","olink_id","beta","se","q",
                "variant_id","slope","slope_se","qval","af","pval_beta"
            ] if c in pqtl_prioritized.columns
        ]
        display(pqtl_prioritized[show].head(100))
else:
    print(
        "\n[CIS-pQTL ACTION] Set CIS_PQTL_RESOURCE to the CDRv9 cis-pQTL summary "
        "resource to generate genetic instrument candidates. The Workbench Data "
        "Dictionary gives the bucket path."
    )

# -------------------------------------------------------------------------
# 6. Aggregate result export — safe outputs only
# -------------------------------------------------------------------------
out = Path("aou_proposal1_v1_outputs")
out.mkdir(exist_ok=True)

# These tables contain protein-level statistical summaries only.
scan.to_csv(out / "proteome_wide_associations.csv", index=False)
hits.to_csv(out / "top_protein_candidates.csv", index=False)

if not pqtl_prioritized.empty:
    pqtl_prioritized.to_csv(out / "pqtl_prioritized_candidates.csv", index=False)

    instrument_cols = [
        c for c in [
            "digital_feature","protein","olink_id","uniprot",
            "variant_id","slope","slope_se","qval","af",
            "beta","se","q"
        ] if c in pqtl_prioritized.columns
    ]
    instruments = pqtl_prioritized[instrument_cols].copy()
    instruments.to_csv(out / "mr_instrument_candidates.tsv", sep="\t", index=False)

# Non-identifying analysis summary.
summary = pd.DataFrame([{
    "proteomics_participants": int(proteomics["person_id"].nunique()),
    "protein_assays": int(proteomics["protein"].nunique()),
    "fitbit_sleep_qc_participants": int(len(sleep)),
    "analysis_overlap_n": int(len(analysis_ids)),
    "association_tests": int(len(scan)),
    "fdr_significant_tests": int((scan["q"] <= FDR_THRESHOLD).sum()),
    "digital_features_tested": ";".join([x for x in DIGITAL_FEATURES if x in sleep.columns]),
    "model_covariates": ";".join(COVARIATES),
}])
summary.to_csv(out / "analysis_summary.csv", index=False)

print("\n" + "="*88)
print("V1 COMPLETE")
print("="*88)
display(summary)
print("\nAggregate result directory:", out.resolve())
print(
    "\nInterpretation note: this V1 is an association/prioritization analysis. "
    "A cis-pQTL-supported protein is a candidate for downstream MR/colocalization, "
    "not proof that the wearable phenotype causally changes that protein."
)
